<a href="https://colab.research.google.com/github/E-Sentinel-Project/E-Sentinel/blob/master/UMAFall.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import files
import zipfile
import os

uploaded = files.upload()
zip_name = list(uploaded.keys())[0]

extract_path = "UMAFall"

with zipfile.ZipFile(zip_name, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

print("Extraction complete.")
print("Top-level folders:")
print(os.listdir(extract_path))


Saving archive (9).zip to archive (9).zip
Extraction complete.
Top-level folders:
['UMAFall_Subject_04_ADL_Aplausing_3_2016-12-03_17-51-44.csv', 'UMAFall_Dataset', 'UMAFall_Subject_03_ADL_MakingACall_2_2017-04-14_23-49-56.csv', 'UMAFall_Subject_05_ADL_LyingDown_OnABed_1_2016-06-15_21-52-40.csv', 'UMAFall_Subject_02_ADL_HandsUp_2_2017-04-26_19-43-25.csv', 'UMAFall_Subject_01_ADL_Aplausing_1_2017-04-14_23-38-23.csv']


In [ ]:
from glob import glob

all_files = glob("UMAFall/**/*.csv", recursive=True)

print("Total CSV files found:", len(all_files))
print("First 10 files:")
print(all_files[:10])


Total CSV files found: 751
First 10 files:
['UMAFall/UMAFall_Subject_04_ADL_Aplausing_3_2016-12-03_17-51-44.csv', 'UMAFall/UMAFall_Subject_03_ADL_MakingACall_2_2017-04-14_23-49-56.csv', 'UMAFall/UMAFall_Subject_05_ADL_LyingDown_OnABed_1_2016-06-15_21-52-40.csv', 'UMAFall/UMAFall_Subject_02_ADL_HandsUp_2_2017-04-26_19-43-25.csv', 'UMAFall/UMAFall_Subject_01_ADL_Aplausing_1_2017-04-14_23-38-23.csv', 'UMAFall/UMAFall_Dataset/UMAFall_Subject_02_Fall_forwardFall_1_2016-06-13_20-43-52.csv', 'UMAFall/UMAFall_Dataset/UMAFall_Subject_13_ADL_LyingDown_OnABed_3_2016-06-06_16-50-42.csv', 'UMAFall/UMAFall_Dataset/UMAFall_Subject_15_ADL_Walking_1_2016-06-17_18-29-58.csv', 'UMAFall/UMAFall_Dataset/UMAFall_Subject_06_ADL_Hopping_3_2016-06-13_21-17-05.csv', 'UMAFall/UMAFall_Dataset/UMAFall_Subject_18_Fall_lateralFall_6_2016-05-29_22-00-46.csv']


In [ ]:
fall_files = [f for f in all_files if "_Fall_" in f]
adl_files = [f for f in all_files if "_ADL_" in f]

print("Fall files:", len(fall_files))
print("ADL files:", len(adl_files))


Fall files: 208
ADL files: 543


In [ ]:
sample_file = fall_files[0]
print("Sample file:", sample_file)

df = pd.read_csv(sample_file)

print("\nColumns:")
print(df.columns)

print("\nFirst 5 rows:")
print(df.head())


Sample file: UMAFall/UMAFall_Dataset/UMAFall_Subject_02_Fall_forwardFall_1_2016-06-13_20-43-52.csv

Columns:
Index(['% Universidad de Malaga - ETSI de Telecomunicacion (Spain)                         '], dtype='object')

First 5 rows:
  % Universidad de Malaga - ETSI de Telecomunicacion (Spain)                         
0  % Date: 2016-06-13_20:43:52                   ...                                 
1  % ID: Subject_02_FALL_forwardFall_1           ...                                 
2  % Name: Subject_02                            ...                                 
3  % Age: 22                                     ...                                 
4  % Height(cm): 167                             ...                                 


In [ ]:
with open(sample_file, 'r') as f:
    lines = f.readlines()

print("First 40 lines:\n")
for i in range(40):
    print(lines[i].strip())


First 40 lines:

% Universidad de Malaga - ETSI de Telecomunicacion (Spain)
% Date: 2016-06-13_20:43:52
% ID: Subject_02_FALL_forwardFall_1
% Name: Subject_02
% Age: 22
% Height(cm): 167
% Weight(Kg): 63
% Gender: F

% Type of Movement: FALL
% Type of Movement: TRUE
% Description of the movement: forwardFall
% Trial: 1

% Number of Sensors: 5

% Used Smartphone: LGE-lge-LG-H815-5.1
% Smartphone's Accelerometer: LGE Accelerometer - Vendor: BOSCH
% --> Version: 1
% --> Min - Max Delay: 5000us - 65535000us
% --> Maximum Range: 16.000000263891405 G
% --> Resolution: 1.2136514986004396E-4 G

% SensorTag's Accelerometer: MPU-9250 MEMS MotionTracking Device - Invensense
% --> Maximum Range: 16 G
% --> Resolution: 0.00024 G

% MAC Address; Sensor_ID; Position; Device Model
%f8:95:c7:f3:ba:82; 0; RIGHTPOCKET; lge-LG-H815-5.1
%C4:BE:84:70:0E:80; 3; WRIST; SensorTag
%C4:BE:84:70:64:8A; 1; CHEST; SensorTag
%B0:B4:48:B8:77:03; 4; ANKLE; SensorTag
%C4:BE:84:71:A5:02; 2; WAIST; SensorTag

% Sensor_Ty

In [ ]:
for i, line in enumerate(lines):
    if not line.startswith("%") and line.strip() != "":
        print("First numeric line index:", i)
        print("Line content:", line.strip())
        break


First numeric line index: 41
Line content: 96;1;0.04825492948293686;0.9503470063209534;0.06907453387975693;0;0


In [ ]:
import numpy as np
import pandas as pd

def load_umafall_file(file_path):

    with open(file_path, 'r') as f:
        lines = f.readlines()

    # Find where numeric data starts
    start_idx = 0
    for i, line in enumerate(lines):
        if not line.startswith("%") and line.strip() != "":
            start_idx = i
            break

    # Read numeric part
    df = pd.read_csv(
        file_path,
        sep=";",
        skiprows=start_idx,
        header=None
    )

    # Columns:
    # 0 = timestamp
    # 1 = sensor_id
    # 2 = x
    # 3 = y
    # 4 = z
    # 5 = sensor_type

    # Keep only accelerometer rows
    df = df[df[5] == 0]

    if len(df) == 0:
        return None

    data = pd.DataFrame({
        "timestamp": df[0].astype(float),
        "x": df[2].astype(float),
        "y": df[3].astype(float),
        "z": df[4].astype(float)
    })

    # Convert timestamp to seconds
    data["timestamp"] = data["timestamp"] - data["timestamp"].iloc[0]
    data["timestamp"] = data["timestamp"] / 1000.0

    return data


In [ ]:
test_data = load_umafall_file(sample_file)

print("Rows loaded:", len(test_data))
print(test_data.head())


Rows loaded: 4176
   timestamp         x         y         z
0        0.0  0.048255  0.950347  0.069075
1        0.0  0.049598  0.946074  0.070296
2        0.0  0.050940  0.947662  0.072249
3        0.0  0.053747  0.947539  0.073347
4        0.0  0.057166  0.946929  0.073468


In [ ]:
class Kalman1D:
    def __init__(self):
        self.x = 0.0
        self.p = 1.0
        self.q = 0.02
        self.r = 0.1

    def update(self, measurement):
        self.p += self.q
        k = self.p / (self.p + self.r)
        self.x += k * (measurement - self.x)
        self.p *= (1 - k)
        return self.x


def normalize(v, min_v, max_v):
    return np.clip((v - min_v) / (max_v - min_v), 0, 1)

def normalizeStableGravity(acc):
    return 1 - np.clip(abs(acc - 9.8) / 5, 0, 1)

def normalizeSmallJerk(j):
    return np.clip(1 - (j / 15), 0, 1)


def detect_fall_neutrosophic(data):

    kFilter = Kalman1D()
    last_acc = np.zeros(3)
    last_ts = None

    predictions = []

    for i in range(len(data)):
        x, y, z, timestamp = data.iloc[i]

        rawAccel = np.sqrt(x*x + y*y + z*z)
        filteredAccel = kFilter.update(rawAccel)

        if last_ts is None:
            dt = 0
            jerk = 0
        else:
            dt = timestamp - last_ts
            diff = np.sqrt((x-last_acc[0])**2 +
                           (y-last_acc[1])**2 +
                           (z-last_acc[2])**2)
            jerk = diff/dt if dt > 0 else 0

        last_ts = timestamp
        last_acc = np.array([x,y,z])

        angle = np.degrees(np.arccos(np.clip(z/filteredAccel, -1, 1)))

        T = (
            normalize(filteredAccel, 15, 30) * 0.55 +
            normalize(jerk, 10, 35) * 0.30 +
            normalize(angle, 35, 85) * 0.15
        )
        T = np.clip(T, 0, 1)

        I = (
            normalize(jerk, 3, 12) * 0.4 +
            normalize(angle, 10, 35) * 0.6
        )
        I = np.clip(I, 0, 0.3)

        F = (
            normalizeStableGravity(filteredAccel) * 0.6 +
            normalizeSmallJerk(jerk) * 0.4
        )
        F = np.clip(F, 0.1, 1)

        predictedFall = (T - F > I) and (T > 0.6)

        predictions.append(1 if predictedFall else 0)

    return predictions


In [ ]:
event_TP = 0
event_FP = 0
event_TN = 0
event_FN = 0

for file in all_files:

    data = load_umafall_file(file)

    if data is None:
        continue

    label = 1 if "_Fall_" in file else 0

    preds = detect_fall_neutrosophic(data)

    detected = any(preds)

    if label == 1:
        if detected:
            event_TP += 1
        else:
            event_FN += 1
    else:
        if detected:
            event_FP += 1
        else:
            event_TN += 1


In [ ]:
print("Event-Level Confusion Matrix:")
print([[event_TN, event_FP],
       [event_FN, event_TP]])

sensitivity = event_TP / (event_TP + event_FN)
specificity = event_TN / (event_TN + event_FP)
f1 = (2 * event_TP) / (2 * event_TP + event_FP + event_FN)

print("\nSensitivity:", round(sensitivity,4))
print("Specificity:", round(specificity,4))
print("F1-score:", round(f1,4))


Event-Level Confusion Matrix:
[[523, 0], [208, 0]]

Sensitivity: 0.0
Specificity: 1.0
F1-score: 0.0


In [ ]:
def load_umafall_file(file_path):

    with open(file_path, 'r') as f:
        lines = f.readlines()

    start_idx = 0
    for i, line in enumerate(lines):
        if not line.startswith("%") and line.strip() != "":
            start_idx = i
            break

    df = pd.read_csv(
        file_path,
        sep=";",
        skiprows=start_idx,
        header=None
    )

    df = df[df[5] == 0]

    if len(df) == 0:
        return None

    data = pd.DataFrame({
        "timestamp": df[0].astype(float),
        "x": df[2].astype(float) * 9.8,
        "y": df[3].astype(float) * 9.8,
        "z": df[4].astype(float) * 9.8
    })

    data["timestamp"] = data["timestamp"] - data["timestamp"].iloc[0]
    data["timestamp"] = data["timestamp"] / 1000.0

    return data


In [ ]:
event_TP = 0
event_FP = 0
event_TN = 0
event_FN = 0

for file in all_files:

    data = load_umafall_file(file)

    if data is None:
        continue

    label = 1 if "_Fall_" in file else 0

    preds = detect_fall_neutrosophic(data)

    detected = any(preds)

    if label == 1:
        if detected:
            event_TP += 1
        else:
            event_FN += 1
    else:
        if detected:
            event_FP += 1
        else:
            event_TN += 1


In [ ]:
print([[event_TN, event_FP],
       [event_FN, event_TP]])

sensitivity = event_TP / (event_TP + event_FN)
specificity = event_TN / (event_TN + event_FP)
f1 = (2 * event_TP) / (2 * event_TP + event_FP + event_FN)

print("Sensitivity:", round(sensitivity,4))
print("Specificity:", round(specificity,4))
print("F1-score:", round(f1,4))


[[355, 168], [58, 150]]
Sensitivity: 0.7212
Specificity: 0.6788
F1-score: 0.5703
